In [ ]:
import jax
import os

import sys
sys.path.append("../")

%load_ext autoreload
%autoreload 2

import pulse_simulator as ps

In [2]:
from datetime import datetime
import functools
import itertools
import logging
import numpy as np
import pickle as pkl

import qiskit as qk
from qiskit.quantum_info import Operator
from qiskit import pulse
import qiskit_aer
import qiskit.providers.fake_provider as qk_fp
import qiskit_dynamics as qk_d

import re 
from scipy.integrate import simpson
from scipy.linalg import sqrtm, expm, logm
from tqdm import tqdm
import warnings

import matplotlib.pyplot as plt

In [ ]:
# Config stuff
jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_platform_name", "cpu")
qk_d.array.Array.set_default_backend("jax")

# Check if GPU is available
jax.default_backend()

# Pulses

In [43]:
# num_qubits = 10

# backend = qk_fp.GenericBackendV2(
#     num_qubits=num_qubits,
#     basis_gates=["id", "rz", "sx", "x", "ecr"],
#     coupling_map=[[i, i + 1] for i in range(num_qubits - 1)],
#     seed=17,
# )

# backend = qk_fp.FakeMumbai()
backend = qk_fp.FakeManila()

In [44]:
Units = 1e9

# Undo units
GHz = 1 / Units
MHz = 1e3 / Units
μs = Units / 1e3
ns = Units

dt = 0.5 # backend.configuration().dt * ns
desired_1qb_duration = 225  # dt
desired_2qb_duration = 225 * 4  # dt

# Gate time in SI units (granularity restriction)
duration_1qb = ps.qiskit_closest_multiple_of_16(desired_1qb_duration * dt / ns, backend)
duration_2qb = ps.qiskit_closest_multiple_of_16(desired_2qb_duration * dt / ns, backend)

In [6]:
# ps.generate_simulation_vars(
#     range(5), 
#     backend.configuration().coupling_map
# )

In [7]:
# ps.backend_simulation_vars(
#     backend,
#     units=1,
#     rabi=False
# )

In [8]:
# dir(backend.configuration())
# backend.configuration().hamiltonian["vars"]

# Pulses

In [ ]:
# Load shapes
fname_1qb = "../pulse_simulator/gatesets/controls_v3_T_120_dt_0.5_a_0.35_dda_0.03.csv"
pico_a = ps.from_csv(fname_1qb, delimiter="\t")
pico_a = np.vstack([pico_a, 1.0 * np.ones(pico_a.shape[1])])

names_1qb = ["X1", "X2", "SX1", "SX2", "ID1", "ID2"]
angles_1qb = [np.pi  / 2, np.pi / 2, np.pi / 4, np.pi / 4, np.pi, 0.0]

cal_1qb_robust = ps.calibrate_gateset(
    gateset=pico_a,
    names=names_1qb,
    angles=angles_1qb,
    duration=duration_1qb,
    dt=dt,
    samples=60
)

# defaults
cal_1qb_default = ps.default_single_qubit_gateset(
    names_1qb,
    duration_1qb,
    dt,
    samples=120
)

In [ ]:
def fname(k):
    return f"../playground/pico-pulses/saved-pulses-2024-01-16/a_two_qubit_{k}.csv"

base_2qb = {k: ps.from_csv(fname(k))[0] for k in ["default", "robust"]}
default_2qb = 2 * [base_2qb["default"]]
# robust_2qb = 2 * [base_2qb["robust"]]
names_2qb = ["+", "-"]

cal_2qb_default = ps.calibrate_gateset(
    gateset=default_2qb,
    names=names_2qb,
    angles=[0.125 * np.pi, -0.125 * np.pi],
    duration=duration_2qb,
    dt=dt,
    samples=32,
)

# cal_2qb_robust = ps.calibrate_gateset(
#     gateset=robust_2qb,
#     names=names_2qb,
#     angles=[0.125 * np.pi, -0.125 * np.pi],
#     duration=duration_2qb,
#     dt=dt,
#     samples=32,
# )

# Circuit

In [ ]:
registers = [0, 1, 2, 3]
ising = ps.TransverseIsingModel(quantum_registers=registers)
# circ = ising.second_order_Trotter_unitary()
circ = ising.first_order_Trotter_unitary()

# registers = [0, 1]
# circ = qk.QuantumCircuit(2, 2)
# [circ.x(i) for i in range(2)]
# circ.cx(0, 1)

circ.draw()

simulator = ps.Simulator(registers, backend)

In [153]:
circ1 = simulator.build_circuit(moments=simulator.build_moments(circ), insert_barriers=True)

In [154]:
new_moments = []
for moment in simulator.build_moments(circ):
    if ps.get_moment_case(moment) == "ecr":
        continue
    else:
        new_moments.append(moment)

circ2 = simulator.build_circuit(moments=new_moments, insert_barriers=True)

In [ ]:
# circ1.draw('mpl')
circ2.draw('mpl')

Random Circuit

In [156]:
from qiskit.circuit.random import random_circuit

In [ ]:
registers = [0, 1]
simulator = ps.Simulator(registers, backend)

depth = 10
rand_circ = random_circuit(len(registers), 17, max_operands=1)
circ3 = simulator.build_circuit(moments=simulator.build_moments(rand_circ), insert_barriers=True)
circ3.draw('mpl')

RB Circuit [[link](https://qiskit-community.github.io/qiskit-experiments/manuals/verification/randomized_benchmarking.html)]

In [ ]:
from qiskit_experiments.library import StandardRB

In [ ]:
registers = [0, 1]
seeds = [17, 18, 19, 20]
simulator = ps.Simulator(registers, backend)
simult_rb = qk.QuantumCircuit(len(registers))

for r, seed in zip(registers, seeds):
    # Create an RB experiment
    lengths = [20]
    rb_exp = StandardRB([0], lengths, num_samples=1, seed=seed)
    rb_circ = qk.transpile(rb_exp.circuits()[0], backend, optimization_level=1)

    # NOTE: update to ps.fn
    rb_circ = remove_final_measurement(remove_idle_wires(rb_circ))
    # rb_circ.draw(output="mpl")
    simult_rb.compose(rb_circ, qubits=r, inplace=True)

simult_rb = simulator.build_circuit(moments=simulator.build_moments(simult_rb), insert_barriers=True)
simult_rb.draw('mpl')

# Simulator

In this example, we loop over the two qubit gatesets.

In [255]:
simulator = ps.Simulator(registers, backend)

fidelities = {} 
cases = [
    "1-default, 2-default",  "1-robust, 2-default",
    # "1-default, 2-robust", "1-robust, 2-robust"
]
gateset_pairs = [
    [cal_1qb_default, cal_2qb_default],
    [cal_1qb_robust, cal_2qb_default],
    # [cal_1qb_default, cal_2qb_robust], 
    # [cal_1qb_robust, cal_2qb_robust], 
]
circ = simult_rb
include_dissipation = True

# --------------------------------

for case, pair in zip(cases, gateset_pairs):
    gateset1, gateset2 = pair
    res = simulator.simulate_circuit(
        circ, gateset1, gateset2, dt, 
        include_crosstalk=True, atol=1e-9, rtol=1e-9,
        include_dissipation=include_dissipation,
        gpu=True
    )

    moments = simulator.get_moments()

    measure_times = np.cumsum(
        ps.durations_of_moments(moments, duration_1qb, duration_2qb)
    ) * dt

    U_goals = [] 
    for i in range(len(moments)):
        U_goals.append(ps.build_operator(moments[0:i+1], registers))

    # skip initial state in result TODO: should we even return it?
    if include_dissipation:
        fids = np.array([
            qk.quantum_info.process_fidelity(R, qk.quantum_info.SuperOp(U))
            for R, U in zip(res[1:], U_goals)
        ])
    else:
        fids = np.array([ps.operator_fidelity(R, U) for R, U in zip(res[1:], U_goals)])
    fidelities[case] = fids

In [243]:
# qk.quantum_info.process_fidelity

In [ ]:
fig, ax = plt.subplots(1)

for case in cases:
    ax.plot(measure_times, 1 - fidelities[case], label=case, marker="o")

# ax.set_yscale("log")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Infidelity")
ax.legend()

In [ ]:
fig, ax = plt.subplots(1)

for case in cases:
    ax.plot(measure_times, 1 - fidelities[case], label=case, marker="o")

# ax.set_yscale("log")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Infidelity")
ax.legend()

This image compares random circuits.

In [ ]:
fig, ax = plt.subplots(1)

for case in cases:
    ax.plot(measure_times, 1 - fidelities[case], label=case, marker="o")

# ax.set_yscale("log")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Infidelity")
ax.legend()

This image shows the TFIM without ECR gates.

In [ ]:
fig, ax = plt.subplots(1)

for case in cases:
    ax.plot(measure_times, 1 - fidelities[case], label=case, marker="o")

# ax.set_yscale("log")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Infidelity")
ax.legend()

This image shows the TFIM with ECR gates and 10x crosstalk.

In [ ]:
fig, ax = plt.subplots(1)

for case in cases:
    ax.plot(measure_times, 1 - fidelities[case], label=case, marker="o")

ax.set_ylim([1e-6, 1.0])
# ax.set_yscale("log")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Infidelity")
ax.legend()

In [ ]:
# The circuit used by the simulator is stored and we can view it
simulator.build_circuit(
    insert_barriers=True,
).draw('mpl')

# The schedules and moments are also in there.